In [4]:
import json

# Load the metadata JSON file
with open('../output_Gated Attention_metadata.json', 'r') as f:
    metadata = json.load(f)

# Extract sections data in the required format: [section_name, [figures, tables, formulas, text_blocks]]
sections_data = []

for section in metadata['sections']:
    section_name = section['original_name']
    stats = section['stats']
    
    # Format: [figures, tables, formulas, text_blocks]
    counts = [
        stats['figures'],
        stats['tables'],
        stats['formulas'],
        stats['text_blocks']
    ]
    
    sections_data.append([section_name, counts])

# Display the result
print("Sections data formatted for agent input:")
print("=" * 60)
for item in sections_data:
    print(item)

# Also store it as a variable for further use
print("\n" + "=" * 60)
print(f"Total sections: {len(sections_data)}")
print("\nVariable 'sections_data' is ready to be used as input to an agent.")

Sections data formatted for agent input:
['Title', [1, 0, 0, 2]]
['Abstract', [0, 0, 0, 1]]
['1 Introduction', [2, 0, 0, 6]]
['2 Gated-Attention Layer', [0, 0, 0, 0]]
['2.1 Preliminary: Multi-Head Softmax Attention', [2, 0, 4, 7]]
['2.2 Augmenting Attention Layer with Gating Mechanisms', [1, 1, 1, 4]]
['3 Experiments', [0, 0, 0, 0]]
['3.1 Experimental Setups', [0, 0, 0, 2]]
['3.2 Main Results', [0, 0, 0, 0]]
['3.2.1 Gated Attention for MoE models', [1, 1, 0, 4]]
['3.2.2 Gated Attention for Dense Models.', [1, 1, 0, 3]]
['4 Analysis: Non-Linearity, Sparsity, and Attention-Sink-Free', [0, 0, 0, 1]]
['4.1 Non-linearity Improves the Expressiveness of Low-Rank Mapping in Attention', [5, 1, 3, 7]]
['4.2 Gating Introduces Input-Dependent Sparsity', [0, 0, 0, 5]]
['4.3 SDPA Output Gating Reduces Attention-Sink', [0, 0, 0, 3]]
['4.4 SDPA Output Gating Facilitates Context Length Extension', [1, 1, 0, 2]]
['5 Related Works', [0, 0, 0, 0]]
['5.1 Gating in Neural Networks', [0, 0, 0, 2]]
['5.2 Atte

In [31]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Literal
import os
import json

from dotenv import load_dotenv
load_dotenv()

# Define Pydantic models following the new schema
class ReadingStep(BaseModel):
    """A single reading step within a pass"""
    step_id: str = Field(..., description="Unique step identifier, e.g., FP-1, SP-2, TP-3")
    name: str = Field(..., description="Brief descriptive name for this step (3-7 words)")
    target_sections: List[str] = Field(
        ...,
        description="Paper sections or subsections this step applies to"
    )
    instruction: str = Field(
        ...,
        description="Action the reader should perform at this step"
    )
    reading_objective: str = Field(
        ...,
        description="What the reader should aim to achieve conceptually"
    )
    reader_checkpoint: str = Field(
        ...,
        description="Observable outcome indicating the step was successfully completed"
    )


class ReadingPass(BaseModel):
    """A single pass in the three-pass reading approach"""
    pass_id: Literal["first_pass", "second_pass", "third_pass"]
    pass_name: str
    objective: str
    estimated_time_minutes: int
    steps: List[ReadingStep]


class StepsGeneratorOutput(BaseModel):
    """Complete reading guide output"""
    paper_id: str
    reading_strategy: Literal["three-pass"] = "three-pass"
    passes: List[ReadingPass]


# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1  # Lower temperature for more focused, consistent output
)

# Prepare context from metadata
paper_title = metadata['paper_title']
abstract = metadata['abstract']
global_stats = metadata['global_stats']
inference = metadata['inference']
paper_id = metadata.get('paper_id', paper_title.replace(' ', '_'))

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert research reading strategist.

Generate a structured reading plan using the three-pass methodology:

**First Pass (5-10 min)**: Quick overview - read title, abstract, introduction, section headings, conclusion. **IMPORTANT: Include a dedicated step to skim ALL figures, tables, and their captions** to identify key visual results.

**Second Pass (~1 hour)**: Read with care but skip deep details. Grasp main thrust. **Pay attention to figures and tables** while reading each section. Note which visuals support key claims. Mark relevant references.

**Third Pass (4-5 hours)**: Deep dive - virtually re-implement the work. **CRITICAL: Include a dedicated step to thoroughly examine ALL formulas, tables, and figures.** Analyze each visual element in detail. Challenge assumptions and verify mathematical derivations.

**Output Requirements:**
- name: Brief, descriptive step name (3-7 words, e.g., "Overview of Paper Structure", "Examine Experimental Results", "Analyze Mathematical Formulations")
- target_sections: Actual section names from the paper structure provided. For steps focusing on figures/tables, list sections that contain them.
- instruction: Specific, actionable reading task. **Create explicit steps for examining figures, tables, and formulas** (e.g., "Skim all figures, tables, and captions", "Examine all tables and figures to evaluate experimental evidence")
- reading_objective: Clear conceptual goal (e.g., "Identify key visual results", "Assess empirical support through visual evidence")
- reader_checkpoint: How to verify completion (e.g., "Can list main findings shown in figures")

**Pass Details:**
- pass_id: "first_pass", "second_pass", "third_pass"
- pass_name: E.g., "First Pass - Overview"
- objective: Overall goal for this pass
- estimated_time_minutes: Realistic estimate (First: 5-10, Second: 45-90, Third: 180-300)

**CRITICAL**: The paper structure includes counts of [figures, tables, formulas, text_blocks] per section. Use this information to prioritize sections with visual content. Always include dedicated steps for examining visual elements."""),
    # ("human", """The following information describes a research paper.  
    ("human", """Generate a three-pass reading plan for this paper:

**Title:** {title}

**Abstract:** {abstract}

**Paper Structure (with [figures, tables, formulas, text_blocks] counts):**
{sections_data}

**Statistics:** {total_pages} pages, {total_sections} sections, **{total_figures} figures, {total_tables} tables**, {total_formulas} formulas

**Classification:** {paper_type} | Difficulty: {difficulty} | Math-heavy: {math_heavy}

---

Create a structured plan with:
- **First Pass** (5-10 min): 3-4 steps for quick overview. **MUST include one step dedicated to skimming all figures and tables.**
- **Second Pass** (45-90 min): 4-6 steps for thorough reading without deep details. Reference figures/tables when reading sections that contain them.
- **Third Pass** (3-5 hours): 3-5 steps for complete understanding and verification. **MUST include dedicated steps for examining all formulas, tables, and figures in detail.**

Use actual section names from the structure above. Prioritize sections with high figure/table counts. Be specific and actionable.""")

])
# Output only the ordered steps grouped by:

# Create structured output chain
structured_llm = llm.with_structured_output(StepsGeneratorOutput)
chain = prompt | structured_llm

# Generate the reading guide
print("Generating 3-pass reading guide...")
print("=" * 80)

result = chain.invoke({
    "title": paper_title,
    "abstract": abstract,
    "sections_data": "\n".join([f"{s[0]}: {s[1]}" for s in sections_data]),
    "total_pages": global_stats['total_pages'],
    "total_sections": global_stats['total_sections'],
    "total_figures": global_stats['total_figures'],
    "total_tables": global_stats['total_tables'],
    "total_formulas": global_stats['total_formulas'],
    "total_text_blocks": global_stats['total_text_blocks'],
    "paper_type": inference['paper_type'],
    "difficulty": inference['difficulty'],
    "math_heavy": inference['math_heavy']
})

# Convert to JSON format
output_json = result.model_dump()

# Save to JSON file
output_filename = f"../output/{paper_id}_reading_guide.json"
with open(output_filename, 'w') as f:
    json.dump(output_json, f, indent=2)

print("\nGenerated Reading Guide (JSON):")
print("=" * 80)
print(json.dumps(output_json, indent=2))

print("\n" + "=" * 80)
total_steps = sum(len(p['steps']) for p in output_json['passes'])
total_time = sum(p['estimated_time_minutes'] for p in output_json['passes'])
print(f"Total passes: {len(output_json['passes'])}")
print(f"Total steps: {total_steps}")
print(f"Total estimated time: {total_time} minutes (~{total_time/60:.1f} hours)")
print(f"\nSaved to: {output_filename}")
print("Reading guide object is stored in 'result' variable.")
print("JSON output is stored in 'output_json' variable.")



Generating 3-pass reading guide...

Generated Reading Guide (JSON):
{
  "paper_id": "Gated Attention for Large Language Models",
  "reading_strategy": "three-pass",
  "passes": [
    {
      "pass_id": "first_pass",
      "pass_name": "First Pass - Overview",
      "objective": "Quick overview of the paper",
      "estimated_time_minutes": 10,
      "steps": [
        {
          "step_id": "FP-1",
          "name": "Read Title and Abstract",
          "target_sections": [
            "Title",
            "Abstract"
          ],
          "instruction": "Read the title and abstract to understand the paper's main topic and contribution",
          "reading_objective": "Understand the paper's main topic and contribution",
          "reader_checkpoint": "Can summarize the paper's main topic and contribution"
        },
        {
          "step_id": "FP-2",
          "name": "Skim Figures and Tables",
          "target_sections": [
            "1 Introduction",
            "2.1 Preliminar